# Supplied evidence and answer fixtures

Use this folder and the shared kernel. Two fictional policy sections are the entire supplied context; no index is needed. All question/response pairs are AI-authored synthetic fixtures, not model observations.

In [ ]:
import json
from pathlib import Path
from pprint import pprint
context = json.loads(Path("docs/dataset/policies.json").read_text())
fixtures = json.loads(Path("docs/dataset/answers.json").read_text())
pprint(context)
pprint([(item["id"], item["question"]) for item in fixtures])

## Read the citation and focused evidence helper

`models/fixtures.py` copies explicit fixture responses. `tools/evidence.py` returns citation-ID decisions and optionally checks only the phrase annual leave is N days against the cited LEAVE-1 numeric field. Read them first; no model runs.

In [ ]:
from models.fixtures import response_fixture
from tools.evidence import check_answer
pprint(response_fixture(fixtures[0]))

## Baseline: citation IDs only

Inspect which check ran, the raw object, and the final exposed answer. Present source IDs do not establish that the answer follows from them.

In [ ]:
baseline = {item["id"]: check_answer(response_fixture(item), context) for item in fixtures}
for name, result in baseline.items():
    pprint({"id": name, **result})

## Abstain and inspect a cited false claim

The policies say nothing about salary increases, so the explicit unsupported fixture abstains honestly. The 200-day answer cites a real ID and passes the baseline despite contradicting the 20-day source.

In [ ]:
pprint(baseline["unsupported"])
pprint({"source": context["LEAVE-1"], "false_but_cited": baseline["wrong_days"]})
assert baseline["unsupported"]["decision"] == "allow"
assert baseline["wrong_days"]["decision"] == "allow"

## Change only check_annual_days on the same inputs

The focused domain check rejects the exact numeric contradiction. It checks neither general entailment nor eligibility and leaves unmatched wording unchecked.

In [ ]:
candidate = {item["id"]: check_answer(response_fixture(item), context, check_annual_days=True) for item in fixtures}
for item in fixtures:
    name = item["id"]
    pprint({"id": name, "baseline": baseline[name]["decision"], "candidate": candidate[name]["decision"], "reason": candidate[name]["reason"], "stages": candidate[name]["stages"], "exposed": candidate[name]["exposed"]})

## The remaining claim is still unsupported

The source mentions full-time employees, not contractor entitlement. The contractor answer remains allowed because it does not match our narrow numeric rule. A valid citation plus this check is not a general truth guarantee.

In [ ]:
assert candidate["normal"]["decision"] == "allow"
assert candidate["wrong_days"]["decision"] == "block"
assert candidate["unchecked_claim"]["decision"] == "allow"
pprint({"remaining_unsupported_claim": candidate["unchecked_claim"], "source": context["LEAVE-1"]})